# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [ ]:
ПРІЗВИЩЕ_ІМЯ = 'Вербицький Юрій'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-31'
НОМЕР        = 3       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 3   ·   Вербицький Юрій   ·   ШІ-31

Ваші вісім стовпців:
   1. номер і/х
   2. помер/жив
   3. Перебіг
   4. Виписка
   5. spO2 без дот
   6. Креат1
   7. Гемогл2
   8. БРзаг2

Ваша пара для аналізу зв’язку: Перебіг та ступінь КТ

Ваше додаткове питання:
   Стовпець «помер/жив» містить числа 1 і 2. Які три різні статистики pandas порахує для нього без жодної скарги — і чому кожна з них безглузда?


In [ ]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,номер і/х,помер/жив,Перебіг,Виписка,spO2 без дот,Креат1,Гемогл2,БРзаг2
0,2377/239,1,важкий,2022-03-10 00:00:00,90.0,84,144,17.0
1,1121/19,2,важкий,2022-03-16 00:00:00,83.0,75.4,137,13.6
2,283/674,2,важкий,2022-01-18 00:00:00,78.0,80.7,105,18.7
3,2372/513,2,важкий,2022-01-13 00:00:00,82.0,168.1,116,13.6
4,1441/671,2,важкий,2022-03-11 00:00:00,80.0,76,124,15.3
5,2573/354,1,важкий,2021-12-26 00:00:00,72.0,130,120,15.3
6,1762/555,2,важкий,2022-05-11 00:00:00,89.0,98,151,15.3
7,2664/348,2,серед важ,2022-05-28 00:00:00,90.0,118,137,15.3


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [ ]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   номер і/х     121 non-null    object 
 1   помер/жив     121 non-null    int64  
 2   Перебіг       121 non-null    object 
 3   Виписка       120 non-null    object 
 4   spO2 без дот  120 non-null    float64
 5   Креат1        117 non-null    object 
 6   Гемогл2       105 non-null    object 
 7   БРзаг2        101 non-null    float64
dtypes: float64(2), int64(1), object(5)
memory usage: 7.7+ KB


In [ ]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- номер і/х
    непорожніх 121 з 121 | унікальних 119 | типи: int×65, str×56
    приклади: ["'2377/239'", "'1121/19'", "'283/674'", "'2372/513'"]

--- помер/жив
    непорожніх 121 з 121 | унікальних 2 | типи: int×121
    приклади: ['np.int64(1)', 'np.int64(2)']

--- Перебіг
    непорожніх 121 з 121 | унікальних 3 | типи: str×121
    приклади: ["'важкий'", "'серед важ'", "'вкрай важкий '"]

--- Виписка
    непорожніх 120 з 121 | унікальних 90 | типи: datetime×118, str×2
    приклади: ['datetime.datetime(2022, 3, 10, 0, 0)', 'datetime.datetime(2022, 3, 16, 0, 0)', 'datetime.datetime(2022, 1, 18, 0, 0)', 'datetime.datetime(2022, 1, 13, 0, 0)']

--- spO2 без дот
    непорожніх 120 з 121 | унікальних 33 | типи: float×120
    приклади: ['np.float64(90.0)', 'np.float64(83.0)', 'np.float64(78.0)', 'np.float64(82.0)']

--- Креат1
    непорожніх 117 з 121 | унікальних 73 | типи: int×104, float×11, str×2
    приклади: ['84', '75.4', '80.7', '168.1']

--- Гемогл2
    непорожніх 105 з 121 | уніка

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [ ]:
ПАСПОРТ = {
    'номер і/х': (
        'ідентифікатор',
        'Унікальний номер історії хвороби слугує лише для позначення пацієнта та не є математичною величиною.'
    ),
    'помер/жив': (
        'номінальна',
        'Числа 1 і 2 є умовними кодами категорій виходу, між якими можливе лише порівняння на рівність або нерівність.'
    ),
    'Перебіг': (
        'порядкова',
        'Значення відображають градацію ступеня важкості захворювання, де є чітке впорядкування, але відстані між рівнями невизначені.'
    ),
    'Виписка': (
        'інтервальна',
        'Дати дозволяють обчислювати різниці та інтервали часових проміжків, проте точковий нуль відліку є повністю умовним.'
    ),
    'spO2 без дот': (
        'відношень',
        'Кількісний показник насичення крові киснем у відсотках має фізичний зміст відношення та природну точку відліку від нуля.'
    ),
    'Креат1': (
        'відношень',
        'Неперервна медична лабораторна величина концентрації з наявним абсолютним нулем, де пропорційні відношення значень мають фізичний сенс.'
    ),
    'Гемогл2': (
        'відношень',
        'Кількісний вимір масової концентрації білка в крові з природним нулем, що дає змогу коректно порівнювати значення у скільки завгодно разів.'
    ),
    'БРзаг2': (
        'відношень',
        'Неперервний показник концентрації загального білірубіну в сироватці крові має фіксований природний нуль та вимірюється в абсолютних одиницях.'
    ),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,номер і/х,ідентифікатор,Унікальний номер історії хвороби слугує лише для позначе...
1,помер/жив,номінальна,"Числа 1 і 2 є умовними кодами категорій виходу, між яким..."
2,Перебіг,порядкова,Значення відображають градацію ступеня важкості захворюв...
3,Виписка,інтервальна,Дати дозволяють обчислювати різниці та інтервали часових...
4,spO2 без дот,відношень,Кількісний показник насичення крові киснем у відсотках м...
5,Креат1,відношень,Неперервна медична лабораторна величина концентрації з н...
6,Гемогл2,відношень,Кількісний вимір масової концентрації білка в крові з пр...
7,БРзаг2,відношень,Неперервний показник концентрації загального білірубіну ...


In [ ]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [ ]:
def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = df[стовпець].copy()
    if шкала.strip().lower() in ('інтервальна', 'відношень', 'абсолютна') and стовпець != 'Виписка':
        з = pd.to_numeric(з, errors='coerce')
    elif стовпець == 'Виписка':
        з = pd.to_datetime(з, errors='coerce')

    з = з.dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш == 'ідентифікатор':
        return р

    m = з.mode()
    р['мода'] = m.iloc[0] if not m.empty else None
    р['частоти'] = dict(з.value_counts())

    if ш in ('порядкова', 'інтервальна', 'відношень', 'абсолютна'):
        if pd.api.types.is_datetime64_any_dtype(з):
            р['медіана'] = з.median().strftime('%Y-%m-%d')
            р['Q25 (q1)'] = з.quantile(0.25).strftime('%Y-%m-%d')
            р['Q75 (q3)'] = з.quantile(0.75).strftime('%Y-%m-%d')
        elif ш == 'порядкова' and not pd.api.types.is_numeric_dtype(з):
            order = ['серед важ', 'важкий', 'вкрай важкий ']
            з_cat = pd.Categorical(з, categories=order, ordered=True)
            р['медіана'] = order[int(np.median(з_cat.codes))]
            р['Q25 (q1)'] = order[int(np.quantile(з_cat.codes, 0.25))]
            р['Q75 (q3)'] = order[int(np.quantile(з_cat.codes, 0.75))]
        else:
            р['медіана'] = float(з.median())
            р['Q25 (q1)'] = float(з.quantile(0.25))
            р['Q75 (q3)'] = float(з.quantile(0.75))

    if ш in ('інтервальна', 'відношень', 'абсолютна'):
        if pd.api.types.is_datetime64_any_dtype(з):
            mean_dt = pd.to_datetime(з.astype('int64').mean())
            std_days = (з - з.min()).dt.total_seconds().std() / (24 * 3600)
            р['середнє'] = mean_dt.strftime('%Y-%m-%d')
            р['ст. відхилення (днів)'] = round(std_days, 2)
        else:
            р['середнє'] = round(float(з.mean()), 2)
            р['ст. відхилення'] = round(float(з.std()), 2)

    if ш in ('відношень', 'абсолютна'):
        if float(з.mean()) != 0:
            р['коеф. варіації (%)'] = round(float(з.std() / з.mean() * 100), 2)

    return р
    # ваш код: додайте моду й частоти для всіх шкал
    # для порядкової — медіану та квантилі
    # для інтервальної та вище — середнє й стандартне відхилення
    # для відношень і абсолютної — коефіцієнт варіації


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- номер і/х  (ідентифікатор)
    {'непорожніх': 121, 'унікальних': 119}

--- помер/жив  (номінальна)
    {'непорожніх': 121, 'унікальних': 2, 'мода': np.int64(1), 'частоти': {1: np.int64(61), 2: np.int64(60)}}

--- Перебіг  (порядкова)
    {'непорожніх': 121, 'унікальних': 3, 'мода': 'важкий', 'частоти': {'важкий': np.int64(90), 'серед важ': np.int64(30), 'вкрай важкий ': np.int64(1)}, 'медіана': 'важкий', 'Q25 (q1)': 'важкий', 'Q75 (q3)': 'важкий'}

--- Виписка  (інтервальна)
    {'непорожніх': 118, 'унікальних': 88, 'мода': Timestamp('2021-09-07 00:00:00'), 'частоти': {Timestamp('2021-09-07 00:00:00'): np.int64(3), Timestamp('2021-09-15 00:00:00'): np.int64(3), Timestamp('2021-09-11 00:00:00'): np.int64(3), Timestamp('2021-11-19 00:00:00'): np.int64(3), Timestamp('2022-06-06 00:00:00'): np.int64(3), Timestamp('2022-04-06 00:00:00'): np.int64(3), Timestamp('2022-01-18 00:00:00'): np.int64(2), Timestamp('2022-05-11 00:00:00'): np.int64(2), Timestamp('2021-11-16 00:00:00'): np.int64(2

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [ ]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
номер і/х,121.0,119.0,2526.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
помер/жив,121.0,NaN,NaN,NaN,1.495868,0.502062,1.0,1.0,1.0,2.0,2.0
Перебіг,121,3,важкий,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Виписка,120,90,2021-11-19 00:00:00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
spO2 без дот,120.0,NaN,NaN,NaN,82.049167,12.208249,0.9,78.75,84.0,89.0,98.0
Креат1,117.0,73.0,76.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Гемогл2,105.0,60.0,144.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
БРзаг2,101.0,NaN,NaN,NaN,16.216832,5.537094,7.4,13.6,15.3,18.7,60.5


In [ ]:
БЕЗГЛУЗДІ = [
    (
        'помер/жив',
        'Середнє арифметичне',
        1.4959,
        'Стовпець є номінальним: значення 1 і 2 — це коди категорій, '
        'тому їхнє середнє не має змістовної інтерпретації.'
    ),
    (
        'помер/жив',
        'Стандартне відхилення',
        0.5021,
        'Стандартне відхилення обчислене для числових кодів номінальних '
        'категорій, тому не характеризує змістовний розкид ознаки.'
    ),
    (
        'номер і/х',
        'Кількість унікальних значень',
        119,
        'Ідентифікатор використовується для розрізнення записів, '
        'а не для вимірювання властивості. Кількість унікальних '
        'номерів не є характеристикою пацієнтів.'
    ),
    (
        'spO2 без дот',
        'Мінімум',
        0.9,
        'Значення 0.9 різко відрізняється від решти значень і може '
        'бути помилкою введення або неправильним форматом запису '
        '(наприклад, замість 90). Потрібна перевірка первинних даних.'
    ),
    (
        'БРзаг2',
        'Максимум',
        60.5,
        'Максимум суттєво перевищує третій квартиль 18.7. Він може '
        'бути викидом або помилкою, тому без перевірки його не можна '
        'вважати репрезентативним значенням.'
    ),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,помер/жив,Середнє арифметичне,1.4959,Стовпець є номінальним: значення 1 і 2 — це коди категор...
1,помер/жив,Стандартне відхилення,0.5021,Стандартне відхилення обчислене для числових кодів номін...
2,номер і/х,Кількість унікальних значень,119.0000,"Ідентифікатор використовується для розрізнення записів, ..."
3,spO2 без дот,Мінімум,0.9000,Значення 0.9 різко відрізняється від решти значень і мож...
4,БРзаг2,Максимум,60.5000,Максимум суттєво перевищує третій квартиль 18.7. Він мож...


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [ ]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Перебіг × ступінь КТ

Перебіг: шкала — порядкова
ступінь КТ: шкала — не у вашому списку

  Перебіг ступінь КТ
0  важкий    сердній
1  важкий   середній
2  важкий  критичний
3  важкий  критичний
4  важкий        NaN
5  важкий   середній


In [ ]:
A, B = 'Перебіг', 'ступінь КТ'

ОБРАНА_МІРА = 'Рангова кореляція Спірмена'

ЧОМУ_САМЕ_ВОНА = (
    'Обидві змінні належать до порядкової шкали вимірювання (порядкова × порядкова), '
    'де важливим є відносне рангування та градація важкості, а не числові відстані між категоріями.'
)

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

# ваш код: обчисліть обрану міру
import pandas as pd
from scipy.stats import spearmanr, pearsonr

perebih_ord = {'серед важ': 1, 'важкий': 2, 'вкрай важкий': 3}

kt_ord = {
    'легкий': 1, 'легка': 1,
    'середній': 2, 'сердній': 2, 'серед ступ': 2, 'серед ступеня': 2,
    'важкий': 3,
    'критичний': 4, 'вкрай важкий': 4
}

data_ord = df[[A, B]].dropna().copy()
data_ord[A] = data_ord[A].astype(str).str.strip().map(perebih_ord)
data_ord[B] = data_ord[B].astype(str).str.strip().map(kt_ord)
data_ord = data_ord.dropna()

rho, p_val_spearman = spearmanr(data_ord[A], data_ord[B])

print(f"Обрана міра ({ОБРАНА_МІРА}):")
print(f"  r_s = {rho:.4f}")
print(f"  p-value = {p_val_spearman:.6f}")


Обрана міра (Рангова кореляція Спірмена):
  r_s = 0.6349
  p-value = 0.000006


In [ ]:
# Тепер — контрприклад: застосуйте кореляцію Пірсона незалежно від шкал
# і поясніть, чому отримане число не можна використовувати.
r_pearson, p_val_pearson = pearsonr(data_ord[A], data_ord[B])

print("\nКонтрприклад (Кореляція Пірсона):")
print(f"  r = {r_pearson:.4f}")
print(f"  p-value = {p_val_pearson:.6f}")



Контрприклад (Кореляція Пірсона):
  r = 0.6326
  p-value = 0.000007


Коефіцієнт Пірсона працює тільки з точними числами, де відома реальна відстань між значеннями. Для категорій («важкий», «середній») ми самі вигадали цифри 1, 2, 3, тому порахований Пірсон вимірює випадкові умовні бонуси, а не справжній медичний зв'язок.

---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [ ]:
ПІДГОТОВКА = {
    'номер і/х': (
        'Вилучити з моделі: ідентифікатор не є інформативною ознакою і спричиняє перенавчання.'
    ),
    'помер/жив': (
        'Бінарне кодування (0/1): категоріальна ознака без порядку, формуємо бінарний прапорець.'
    ),
    'Перебіг': (
        'Ordinal Encoding (0, 1, 2): зберігаємо чітку градацію важкості захворювання.'
    ),
    'Виписка': (
        'Перетворення на кількість днів від старт-дати: перехід від календарної дати до числового інтервалу.'
    ),
    'spO2 без дот': (
        'Z-score стандартизація: нормування неперервної величини до єдиного масштабу.'
    ),
    'Креат1': (
        'Очищення та Z-score стандартизація: каст до float та масштабування неперервного показника.'
    ),
    'Гемогл2': (
        'Очищення та Z-score стандартизація: каст до float та масштабування неперервного показника.'
    ),
    'БРзаг2': (
        'Логарифмування (log1p) + Z-score: згладжування правосторонньої асиметрії вибірки та нормування.'
    ),
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,номер і/х,ідентифікатор,Вилучити з моделі: ідентифікатор не є інформативною озна...
1,помер/жив,номінальна,Бінарне кодування (0/1): категоріальна ознака без порядк...
2,Перебіг,порядкова,"Ordinal Encoding (0, 1, 2): зберігаємо чітку градацію ва..."
3,Виписка,інтервальна,Перетворення на кількість днів від старт-дати: перехід в...
4,spO2 без дот,відношень,Z-score стандартизація: нормування неперервної величини ...
5,Креат1,відношень,Очищення та Z-score стандартизація: каст до float та мас...
6,Гемогл2,відношень,Очищення та Z-score стандартизація: каст до float та мас...
7,БРзаг2,відношень,Логарифмування (log1p) + Z-score: згладжування правостор...


In [ ]:
# ваш код: реалізуйте підготовку для трьох стовпців різних шкал
import pandas as pd
import numpy as np

df_prepared = pd.DataFrame(index=df.index)

perebih_map = {'серед важ': 0, 'важкий': 1, 'вкрай важкий': 2}
df_prepared['Перебіг_encoded'] = df['Перебіг'].astype(str).str.strip().map(perebih_map)

df_prepared['помер_жив_encoded'] = df['помер/жив'].map({1: 0, 2: 1})

spo2_numeric = pd.to_numeric(df['spO2 без дот'], errors='coerce')
std_spo2 = spo2_numeric.std()
assert std_spo2 > 0

df_prepared['spO2_std'] = (spo2_numeric - spo2_numeric.mean()) / std_spo2

print(df_prepared.head(8))

   Перебіг_encoded  помер_жив_encoded  spO2_std
0                1                  0  0.651267
1                1                  1  0.077884
2                1                  1 -0.331675
3                1                  1 -0.004027
4                1                  1 -0.167851
5                1                  0 -0.823146
6                1                  1  0.569355
7                0                  1  0.651267


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [ ]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [ ]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [ ]:
ВИПРАВЛЕННЯ = [
    (
        1,
        "Обчислення середнього арифметичного для номінальної шкали",
        "Стовпець містить умовні категоріальні коди (1-помер, 2-жив). Середнє 1.50 є штучним математичним числом і не означає «середину між життям і смертю».",
        "Для номінальної шкали слід наводити частоти та відсотки кожної категорії (померло: 61, вижило: 60)."
    ),
    (
        2,
        "Неочищені текстові дані сприйнято як 4 різні статі через дублювання позначень",
        "Числа 56, 34, 19, 12 отримані через некоректне розбиття єдиних категорій «ж/жін» та «ч/чол», що викривляє реальні демографічні пропорції вибірки.",
        "Привести текстові записи до єдиного формату («чол»/«жін» або «ч»/«ж») і порахувати частоти за 2 статями."
    ),
    (
        3,
        "Алфавітне сортування та обчислення середнього для неочищеної порядкової шкали",
        "Алфавітне сортування не відображає медичний порядок важкості, а кодування непідготованого тексту дає безглузде середнє 5.9, яке не має клінічного сенсу.",
        "Стандартизувати назви категорій, впорядкувати за зростанням важкості та обчислити моду або медіану."
    ),
    (
        4,
        "Обчислення статистик поверх непідготованого стовпця зі змішаним типом даних",
        "Стовпець містить діапазони («20-30%») та десяткові дроби («0.8»), тому функція to_numeric здебільшого повернула NaN, а середнє 39.6% обчислено на спотворених даних.",
        "Очистити дані (привести відсотки й діапазони до єдиного числового формату) і тільки потім рахувати середнє."
    ),
    (
        5,
        "Включення явного артефакту/описки в розрахунок загальних статистик",
        "Значення сатурації 0.9 є явним аномальним викидом (опискою введення 90%), яке штучно занижує мінімум та спотворює середнє значення сатурації вибірки.",
        "Виправити описку на 90.0 або вилучити артефакт перед розрахунком описових статистик."
    ),
    (
        6,
        "Обчислення коефіцієнта варіації для дат від абсолютно умовної точки відліку Unix epoch",
        "Коефіцієнт варіації менше 1% є хибним артефактом ділення на величезне середнє значення днів від 1970 року і не відображає реальну варіативність часового потоку.",
        "Розраховувати варіативність часового ряду в реальних днях (тривалостях) відносно першої госпіталізації у вибірці."
    ),
    (
        7,
        "Застосування кореляції Пірсона до якісних номінальних категорій",
        "Кореляція Пірсона вимагає неперервних числових шкал. Коефіцієнт 0.177 для штучних кодів 0 і 1 не вимірює лінійний зв'язок і не підтверджує його відсутність.",
        "Побудувати таблицю спряженості та використати критерій Хі-квадрат або коефіцієнт V Крамера."
    ),
    (
        8,
        "Нормалізація min-max для стовпців із нульовою дисперсією (констант) та пропущеними даними",
        "Стовпець ФЕРИТ2 містить однакові значення або суцільні пропуски, через що ділення на нульовий діапазон утворило повністю порожній стовпець із NaN.",
        "Перед нормалізацією перевіряти варіативність ознак (std > 0) і вилучати константні стовпці."
    ),
    (
        9,
        "Застосування dropna() до всього датасету із великою кількістю пропусків у різних стовпцях",
        "Видалення всіх рядків із пропускними значеннями призвело до втрати 100% спостережень (0 рядків), роблячи датасет непридатним для моделювання.",
        "Застосовувати заповнення пропусків (imputation) або видаляти лише ті стовпці/рядки, де пропусків занадто багато."
    ),
    (
        10,
        "Застосування витоку даних (Data Leakage) та кореляції до дат розтину",
        "Дата розтину заповнюється лише після смерті пацієнта (є результатом, а не предиктором). Кореляція r=1.0 відображає факт смерті, а не фактор ризику.",
        "Повністю вилучити дату розтину з переліку предикторів, оскільки це прямий виток цільової змінної."
    )
]


assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,Обчислення середнього арифметичного для номінальної шкали,"Стовпець містить умовні категоріальні коди (1-помер, 2-ж...",Для номінальної шкали слід наводити частоти та відсотки ...
1,2,Неочищені текстові дані сприйнято як 4 різні статі через...,"Числа 56, 34, 19, 12 отримані через некоректне розбиття ...",Привести текстові записи до єдиного формату («чол»/«жін»...
2,3,Алфавітне сортування та обчислення середнього для неочищ...,Алфавітне сортування не відображає медичний порядок важк...,"Стандартизувати назви категорій, впорядкувати за зростан..."
3,4,Обчислення статистик поверх непідготованого стовпця зі з...,Стовпець містить діапазони («20-30%») та десяткові дроби...,Очистити дані (привести відсотки й діапазони до єдиного ...
4,5,Включення явного артефакту/описки в розрахунок загальних...,Значення сатурації 0.9 є явним аномальним викидом (описк...,Виправити описку на 90.0 або вилучити артефакт перед роз...
5,6,Обчислення коефіцієнта варіації для дат від абсолютно ум...,Коефіцієнт варіації менше 1% є хибним артефактом ділення...,Розраховувати варіативність часового ряду в реальних дня...
6,7,Застосування кореляції Пірсона до якісних номінальних ка...,Кореляція Пірсона вимагає неперервних числових шкал. Кое...,Побудувати таблицю спряженості та використати критерій Х...
7,8,Нормалізація min-max для стовпців із нульовою дисперсією...,Стовпець ФЕРИТ2 містить однакові значення або суцільні п...,Перед нормалізацією перевіряти варіативність ознак (std ...
8,9,Застосування dropna() до всього датасету із великою кіль...,Видалення всіх рядків із пропускними значеннями призвело...,Застосовувати заповнення пропусків (imputation) або вида...
9,10,Застосування витоку даних (Data Leakage) та кореляції до...,Дата розтину заповнюється лише після смерті пацієнта (є ...,"Повністю вилучити дату розтину з переліку предикторів, о..."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [ ]:
# ваш код: три правильні обчислення замість хибних
import pandas as pd
import numpy as np


# 1. РОЗДІЛ 1.
помилковий_результат = df['помер/жив'].mean()

коректний_результат_абс = df['помер/жив'].map({1: 'помер', 2: 'жив'}).value_counts()
коректний_результат_відс = df['помер/жив'].map({1: 'помер', 2: 'жив'}).value_counts(normalize=True) * 100

print("--- РОЗДІЛ 1: Результат лікування ---")
print(f"ХИБНО:  Середнє значення = {помилковий_результат:.2f} (не має клінічного змісту)")
print("ПРАВИЛЬНО: Частоти виходу пацієнтів:")
for статус, абс in коректний_результат_абс.items():
    відс = коректний_результат_відс[статус]
    print(f"  • {статус}: {абс} осіб ({відс:.1f}%)")
print()


# 2. РОЗДІЛ 2. Демографія (Стать)

помилкова_кількість_статей = df['Стать'].nunique()

стать_чиста = df['Стать'].astype(str).str.strip().str[0].map({'ч': 'чоловіча', 'ж': 'жіноча'})
коректна_кількість_статей = стать_чиста.nunique()
коректний_розподіл_статі = стать_чиста.value_counts()

print("--- РОЗДІЛ 2: Розподіл за статтю ---")
print(f"ХИБНО:  Знайдено {помилкова_кількість_статей} статі (через дублювання позначень)")
print(f"ПРАВИЛЬНО: Реальних статей: {коректна_кількість_статей}")
for стать, кількість in коректний_розподіл_статі.items():
    print(f"  • {стать}: {кількість} осіб")
print()


# 3. РОЗДІЛ 4. Обсяг ураження легень
кт_хибне = pd.to_numeric(df['КТ'], errors='coerce')

print("--- РОЗДІЛ 4: Обсяг ураження легень (%) ---")
print(
    f"ХИБНО:  Враховано лише {кт_хибне.notna().sum()} числових записів з 121 (інші 17 діапазонів втрачено)"
)
print(f"  • Хибне середнє: {round(100 * кт_хибне.mean(), 1)}%\n")


def parse_kt_percent(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip().replace('%', '').replace('до ', '')
    if '-' in val_str:
        parts = val_str.split('-')
        try:
            return (float(parts[0]) + float(parts[1])) / 2.0
        except ValueError:
            return np.nan
    try:
        num = float(val_str)
        return num * 100.0 if num <= 1.0 else num
    except ValueError:
        return np.nan


кт_чисте = df['КТ'].apply(parse_kt_percent)

print(
    f"ПРАВИЛЬНО: Опрацьовано всі {кт_чисте.notna().sum()} дійсних записів КТ:"
)
print(f"  • Середнє обсягу ураження: {round(кт_чисте.mean(), 1)}%")
print(f"  • Медіана:                 {round(кт_чисте.median(), 1)}%")
print(f"  • Стандартне відхилення:   {round(кт_чисте.std(), 1)}%")

--- РОЗДІЛ 1: Результат лікування ---
ХИБНО:  Середнє значення = 1.50 (не має клінічного змісту)
ПРАВИЛЬНО: Частоти виходу пацієнтів:
  • помер: 61 осіб (50.4%)
  • жив: 60 осіб (49.6%)

--- РОЗДІЛ 2: Розподіл за статтю ---
ХИБНО:  Знайдено 4 статі (через дублювання позначень)
ПРАВИЛЬНО: Реальних статей: 2
  • жіноча: 75 осіб
  • чоловіча: 46 осіб

--- РОЗДІЛ 4: Обсяг ураження легень (%) ---
ХИБНО:  Враховано лише 27 числових записів з 121 (інші 17 діапазонів втрачено)
  • Хибне середнє: 39.6%

ПРАВИЛЬНО: Опрацьовано всі 43 дійсних записів КТ:
  • Середнє обсягу ураження: 42.5%
  • Медіана:                 35.0%
  • Стандартне відхилення:   22.0%


---
# Етап 7. Висновки та індивідуальне питання

In [ ]:
ВИСНОВКИ = [
    (
        "Аналіз даних показав майже рівний розподіл пацієнтів за результатами лікування (61 померлий, 50.4% проти 60 вижилих, 49.6%), а середній рівень сатурації кисню без дотації становить 82.05%, що свідчить про важкий стан більшості госпіталізованих."
    ),
    (
        "Для двох порядкових показників «Перебіг» та «ступінь КТ» виявлено прямий і статистично значущий зв'язок (кореляція Спірмена r_s = 0.6349, p < 0.001), що підтверджує відповідність важкості хвороби реальному ураженню легень."
    ),
    (
        "Лабораторні показники БРзаг2 (середнє 16.22) та Креат1 (середнє 108.14) мають високий розкид у вибірці (коефіцієнти варіації 34.14% та 32.65%), тому перед побудовою моделей їх потрібно прологарифмувати та стандартизувати за Z-score."
    ),
]


assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. Аналіз даних показав майже рівний розподіл пацієнтів за результатами лікування (61 померлий, 50.4% проти 60 вижилих, 49.6%), а середній рівень сатурації кисню без дотації становить 82.05%, що свідчить про важкий стан більшості госпіталізованих.
2. Для двох порядкових показників «Перебіг» та «ступінь КТ» виявлено прямий і статистично значущий зв'язок (кореляція Спірмена r_s = 0.6349, p < 0.001), що підтверджує відповідність важкості хвороби реальному ураженню легень.
3. Лабораторні показники БРзаг2 (середнє 16.22) та Креат1 (середнє 108.14) мають високий розкид у вибірці (коефіцієнти варіації 34.14% та 32.65%), тому перед побудовою моделей їх потрібно прологарифмувати та стандартизувати за Z-score.


In [ ]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = """
Стовпець «помер/жив» належить до номінальної шкали вимірювання, де числа 1 і 2 є лише умовними кодами категорій (1 — помер, 2 — жив), а не математичними величинами.
pandas без жодних скарг чи помилок обчислить для нього такі три статистики:
1. Середнє арифметичне (mean ≈ 1.50): Число 1.5 не має жодного клінічного змісту, адже людина не може бути "напівживою" або "середньою між життям і смертю". Це число не вимірює рівень смертності безпосередньо.
2. Стандартне відхилення (std ≈ 0.50): Вимірює математичний розкид значень відносно безглуздого середнього. Оскільки чисельний інтервал між 1 і 2 є суто умовним, оцінювати варіативність числовим відхиленням некоректно.
3. Медіану та квартилі (median / quantile): Для номінальної шкали не існує природного порядкового ранжування. Якщо змінити коди місцями (наприклад, 1 — жив, 2 — помер), медіана зміниться, хоча самі дані залишаться тими ж.
Для номінальних даних єдиними допустимими статистиками є мода, частоти та відсоткові частки кожної категорії.
"""
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Стовпець «помер/жив» містить числа 1 і 2. Які три різні статистики pandas порахує для нього без жодної скарги — і чому кожна з них безглузда?


Стовпець «помер/жив» належить до номінальної шкали вимірювання, де числа 1 і 2 є лише умовними кодами категорій (1 — помер, 2 — жив), а не математичними величинами. 
pandas без жодних скарг чи помилок обчислить для нього такі три статистики:
1. Середнє арифметичне (mean ≈ 1.50): Число 1.5 не має жодного клінічного змісту, адже людина не може бути "напівживою" або "середньою між життям і смертю". Це число не вимірює рівень смертності безпосередньо.
2. Стандартне відхилення (std ≈ 0.50): Вимірює математичний розкид значень відносно безглуздого середнього. Оскільки чисельний інтервал між 1 і 2 є суто умовним, оцінювати варіативність числовим відхиленням некоректно.
3. Медіану та квартилі (median / quantile): Для номінальної шкали не існує природного порядкового ранжування. Якщо змінити коди місцями (наприклад, 1 — жив, 2 — помер), ме

---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [ ]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'текст, перероблено',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'ні',
    'етап 4 — міра зв’язку':           'синтаксис',
    'етап 5 — підготовка до моделі':   'синтаксис',
    'етап 6 — виправлення аналізу':    'текст, перероблено',
    'етап 7 — висновки й відповідь':   'текст, перероблено',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'Перевірив відповідність статистичних показників типам шкал, перерахував кореляцію Спірмена для порядкових змінних, очистив відсотки КТ та переперевірив розрахунки частот для номінальних даних у 6 розділі."'
ПІДТВЕРДЖУЮ = True     # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [ ]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 3 готова до захисту.
Вербицький Юрій, ШІ-31


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.